# Sentimental Analysis using RNN

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("IMDB_Dataset.csv")

In [ ]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Data Cleaning

In [ ]:
data.shape

(50000, 2)

In [ ]:
data.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
data.duplicated().sum()

np.int64(418)

In [ ]:
data.drop_duplicates(inplace=True) # Removing duplicates values from data

In [ ]:
data.shape

(49582, 2)

## Pre-processing

### Text Pre-processing

1. Convert every text into lowercase.
2. Remove URL's (HTTP,HTTPS,etc).
3. Remove HTML Tags.
4. Remove Punctuation (",";"?";"#").
5. Remove Stopwords.
6. Stemming
7. Target values Encoding. (Sentiment)
8. Vectorization. (converting text into Numbers)



In [ ]:
# Converting to Lowercase

data["review"] = data["review"].str.lower()

In [ ]:
# Remove URL's
import re

def remove_urls(text):
  re.sub(r"http\S+","",text)  # (pattern, replacement, string)
  return text

data["review"] = data["review"].apply(remove_urls)

In [ ]:
# Remove punctuation

def remove_punctuations(text):
  text = re.sub(r"[^A-Za-z0-9\s]","",text) # we are using [^A-Za-z0-9\s] to tell regex that we are excluding this and remove everything else.
  return text

data["review"] = data["review"].apply(remove_punctuations)

In [ ]:
# Remove HTML Tags

def remove_html_tags(text):
  text = re.sub(r"<.*?>","",text) # . -> any character; * -> 0 or more character; ? -> remove using non greedy manner (whenever we get first ">" then remove it)
  return text

data["review"] = data["review"].apply(remove_html_tags)

In [ ]:
# removing stopwords

import nltk
nltk.download("punkt") # official tokenizer of nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
# sample_text = "I am Iron Man !"
# tokens = word_tokenize(sample_text)
# print(tokens)

In [ ]:
def remove_stopwords(text):
  tokens = word_tokenize(text)
  stop_words = stopwords.words("english")

  text = " " + text + " "

  for word in tokens:
    if word in stop_words:
      text = text.replace(" " + word + " "," ")

  return text.strip()

data["review"] = data["review"].apply(remove_stopwords)

In [ ]:
data.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [ ]:
# Stemming (Porter Stemming)

from nltk.stem import PorterStemmer

def stemming(text):
  pt = PorterStemmer()
  stemmed_words = []
  tokens = word_tokenize(text)

  for token in tokens:
    stemmed_token = pt.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)

data["review"] = data["review"].apply(stemming)

In [ ]:
data.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [ ]:
# Encoding target value

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data["sentiment"] = le.fit_transform(data["sentiment"])
Y = data["sentiment"]

In [ ]:
# Vectorization using TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=6000)

X = tf.fit_transform(data["review"])

In [ ]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4197425 stored elements and shape (49582, 6000)>

In [ ]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4197425 stored elements and shape (49582, 6000)>
  Coords	Values
  (0, 3741)	0.022128540467174104
  (0, 4428)	0.05221143071521605
  (0, 3367)	0.05546283997603395
  (0, 5803)	0.08549347119935949
  (0, 3820)	0.4705737524247224
  (0, 1835)	0.11187163565261775
  (0, 5986)	0.057569432716137035
  (0, 2591)	0.07797786088408613
  (0, 4456)	0.08983144519317353
  (0, 1891)	0.06091702213199625
  (0, 2441)	0.044845054225427394
  (0, 3342)	0.08080404548641136
  (0, 704)	0.06502368733824271
  (0, 2066)	0.06763275395829006
  (0, 5372)	0.0343574448497755
  (0, 5143)	0.17624923055718872
  (0, 759)	0.07215627612111643
  (0, 4623)	0.033193587673225756
  (0, 5732)	0.24967622138891002
  (0, 4723)	0.043542598383064
  (0, 5926)	0.053200497502287294
  (0, 2307)	0.03316827157616061
  (0, 5533)	0.0747581832125783
  (0, 4794)	0.14381290292097462
  (0, 1959)	0.10012441244029888
  :	:
  (49581, 1975)	0.18025025443742587
  (49581, 2622)	0.088083814477965

## Dataset and Data Loaders

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

(39665, 6000)

In [ ]:
X_test.shape

(9917, 6000)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch

In [ ]:
X_train # this is of type sparse matrix and we cannot use sparse matrix directly into dataset and data loader

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3359836 stored elements and shape (39665, 6000)>

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()  # inorder to use them we need to convert them from sparse matrix to numpy array

In [ ]:
train_dataset =TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(Y_train.values).float()
)

test_dataset =TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(Y_test.values).float()
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 64,
    shuffle = True
)

## Building RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# This is an "Many to One" Architecture

class RNN(nn.Module):

  def __init__(self, input_size, hidden_size=128, num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN layer
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

    # Fully connected layer
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    # this is an optional step => shape of h0 -> (num of layers, batch size, hidden size)
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=device)

    out, _ = self.rnn(x, h0)
    # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
    # 2nd value = final hidden state of last timestamp

    out = self.fc(out[:,-1,:])

    return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)
model = model.to(device)
loss = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

Note:- In PYtorch there are two function used for dimension changing.
1. Unsqueeze (add 1 new dimension)
2. Squeeze (reduce by one dimension)



In [ ]:
epochs = 12
best_val_loss = float("inf")

for epoch in range(epochs):
  model.train()

  for xb, yb in train_loader:
    xb = xb.to(device)
    yb = yb.to(device)
    optimizer.zero_grad()

    xb = xb.unsqueeze(1) # add singleton direction / add extra 1 dimension

    out = model(xb) # output shape => (batch_size,1), but these are not probability

    out = torch.sigmoid(out.squeeze()) # use squeeze to (batch_size,) then sigmod to convert it into probability

    loss_val = loss(out,yb)
    loss_val.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{epochs} | Loss: {loss_val.item()}")

  if loss_val.item() < best_val_loss:
    best_val_loss = loss_val.item()
    torch.save(model.state_dict(), "best_model.pth")
    print(f"  --> Saved new best model with test loss: {best_val_loss:.4f}")

Epoch: 1/12 | Loss: 0.299989253282547
  --> Saved new best model with test loss: 0.3000
Epoch: 2/12 | Loss: 0.17370451986789703
  --> Saved new best model with test loss: 0.1737
Epoch: 3/12 | Loss: 0.1655314713716507
  --> Saved new best model with test loss: 0.1655
Epoch: 4/12 | Loss: 0.2232193797826767
Epoch: 5/12 | Loss: 0.15665178000926971
  --> Saved new best model with test loss: 0.1567
Epoch: 6/12 | Loss: 0.23005907237529755
Epoch: 7/12 | Loss: 0.5189732909202576
Epoch: 8/12 | Loss: 0.4151175916194916
Epoch: 9/12 | Loss: 0.2523784935474396
Epoch: 10/12 | Loss: 0.13212300837039948
  --> Saved new best model with test loss: 0.1321
Epoch: 11/12 | Loss: 0.09731325507164001
  --> Saved new best model with test loss: 0.0973
Epoch: 12/12 | Loss: 0.14529497921466827


## Evaluation

In [ ]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [ ]:
model.eval()

with torch.no_grad():
  correct_val = 0
  total = 0

  for xb, yb in test_loader:
    xb = xb.to(device)
    yb = yb.to(device)

    xb = xb.unsqueeze(1)

    out = model(xb)
    predicted = (torch.sigmoid(out.squeeze()) > 0.5).float()

    total += yb.size(0)
    correct_val += (predicted == yb).sum().item()

  print(f"Total: {total}")
  print(f"Correct: {correct_val}")
  print(f"Accuracy: {correct_val/total*100} %")

Total: 9917
Correct: 8611
Accuracy: 86.83069476656246 %
